# Gold — Camada Analítica (`case.gold`)

Modelo dimensional em estrela: **5 dimensões** e **1 tabela fato**. Timestamps da `tb_chamados_hora` como **fonte de verdade** para tempos de espera e atendimento.

In [0]:
from pyspark.sql import functions as F

catalog = "case"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"Schema {gold_schema} garantido.")

## Arquitetura dimensional

```
           dim_cliente         dim_atendente
              (47K)               (25)
                 |                    |
  dim_tempo — ft_chamados — dim_canal
   (jan25-jul26)  (131.838)     (4)
                    |             
               dim_motivo        bridge_chamado_pesquisa
                 (14)           (81.021, N:N)
```

### Decisões de ouro

| Decisão | Escolha | Justificativa |
|---|---|---|
| Fonte de verdade para tempos | `tb_chamados_hora` (timestamps) | Campos calculados da tb_chamados divergem em 21% dos casos (PDF: "cabe ao grupo decidir") |
| Granularidade do fato | 1 linha por chamado | Um chamado = um evento de atendimento |
| Bridge de pesquisa | N:N chamado ↔ pesquisa | 802 chamados têm múltiplas pesquisas; preservar para auditoria |
| Idade inválida | NULL quando `status_idade ≠ ok` | Evita distorcer análises demográficas |
| Chaves | Naturais (id_cliente, id_motivo, etc.) | Dimensões pequenas, sem SCD |

### Estratégia de joins na fato

| Tabela | Tipo | Motivo |
|---|---|---|
| tb_chamados_hora | INNER | Base 1:1, mesma origem |
| tb_motivos | LEFT (via `nome_motivo_normalizado`) | FK `id_motivo` |
| tb_canais | LEFT (via `nome_canal`) | FK `id_canal` |
| tb_clientes | LEFT | Validar FK `id_cliente` (2.363 sem cliente) |
| tb_custos | LEFT | Medida `custo` (1.550 sem custo) |
| tb_pesquisa_satisfacao_chamado | LEFT | Medida `nota_atendimento` (58.455 sem nota útil) |
| tb_base_atendentes | LEFT | Validar FK `id_atendente` (764 com -1) |

**Idempotência:** todas as gravações usam `mode("overwrite")` + `overwriteSchema`.

In [0]:
# --- dim_cliente ---
# Idade NULL quando status_idade ≠ "ok" (455 nulos + 244 zero + 131 >120 na origem).
df_dim_cliente = spark.table(f"{silver_schema}.tb_clientes").select(
    F.col("id_cliente"),
    F.col("nome"),
    F.col("email"),
    F.col("regiao"),
    F.when(F.col("status_idade") == "ok", F.col("idade")).otherwise(None).alias("idade"),
    F.col("data_cadastro"),
    F.col("dia_vencimento_fatura"),
    F.col("status_idade"),
)

df_dim_cliente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_cliente")
print(f"dim_cliente: {df_dim_cliente.count():,} linhas")

In [0]:
# --- dim_atendente ---
df_dim_atendente = spark.table(f"{silver_schema}.tb_base_atendentes")

df_dim_atendente.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_atendente")
print(f"dim_atendente: {df_dim_atendente.count()} linhas")

In [0]:
# --- dim_canal ---
df_dim_canal = spark.table(f"{silver_schema}.tb_canais")

df_dim_canal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_canal")
print(f"dim_canal: {df_dim_canal.count()} linhas")

In [0]:
# --- dim_motivo ---
df_dim_motivo = spark.table(f"{silver_schema}.tb_motivos")

df_dim_motivo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_motivo")
print(f"dim_motivo: {df_dim_motivo.count()} linhas")

In [0]:
# --- dim_tempo ---
# Gerada a partir do range de datas dos chamados (jan/2025 a jul/2026).
df_data = spark.table(f"{silver_schema}.tb_chamados_hora").select(
    F.to_date(F.col("hora_abertura_chamado")).alias("data")
).distinct()

df_dim_tempo = df_data.select(
    F.col("data"),
    F.year("data").alias("ano"),
    F.month("data").alias("mes_num"),
    F.date_format("data", "MMMM").alias("mes_nome"),
    F.dayofmonth("data").alias("dia"),
    F.dayofweek("data").alias("dia_semana_num"),
    F.date_format("data", "EEEE").alias("dia_semana_nome"),
    F.date_format("data", "yyyy-MM").alias("mes_ano"),
    F.quarter("data").alias("trimestre"),
    F.when(F.month("data") <= 6, 1).otherwise(2).alias("semestre"),
)

df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_tempo")
print(f"dim_tempo: {df_dim_tempo.count()} linhas")

In [0]:
# --- Leitura das tabelas Silver ---
df_chamados = spark.table(f"{silver_schema}.tb_chamados")
df_chamados_hora = spark.table(f"{silver_schema}.tb_chamados_hora")
df_clientes = spark.table(f"{silver_schema}.tb_clientes")
df_custos = spark.table(f"{silver_schema}.tb_custos")
df_pesquisa = spark.table(f"{silver_schema}.tb_pesquisa_satisfacao_chamado")
df_motivos = spark.table(f"{silver_schema}.tb_motivos")
df_canais = spark.table(f"{silver_schema}.tb_canais")
df_atendentes = spark.table(f"{silver_schema}.tb_base_atendentes")

# --- Build da tabela fato ft_chamados ---
# Decisão de ouro: tb_chamados_hora é a fonte de verdade para timestamps.
# tempo_espera e tempo_atendimento são recalculados a partir dos timestamps.
df_gold = (
    df_chamados.alias("c")
    .join(df_chamados_hora.alias("ch"), on="id_chamado", how="inner")
    .join(df_motivos.alias("m"),
          on=F.col("c.motivo") == F.col("m.nome_motivo_normalizado"), how="left")
    .join(df_canais.alias("can"),
          on=F.col("c.canal") == F.col("can.nome_canal"), how="left")
    .join(df_clientes.alias("cli"), on="id_cliente", how="left")
    .join(df_custos.alias("cu"), on="id_chamado", how="left")
    .join(df_pesquisa.alias("ps"), on="id_chamado", how="left")
    .join(df_atendentes.alias("a"),
          on=F.col("c.id_atendente") == F.col("a.id_atendente"), how="left")
    .select(
        F.col("c.id_chamado"),
        F.col("c.id_cliente"),
        F.col("c.motivo"),
        F.col("c.canal"),
        F.col("c.resolvido"),
        F.col("c.id_atendente"),
        # FKs para dimensoes
        F.col("m.id_motivo"),   # FK dim_motivo
        F.col("can.id_canal"),  # FK dim_canal
        # Timestamps (fonte de verdade: tb_chamados_hora)
        F.col("ch.hora_abertura_chamado"),
        F.col("ch.hora_inicio_atendimento"),
        F.col("ch.hora_finalizacao_atendimento"),
        F.col("ch.status_consistencia"),
        # Tempos calculados a partir dos timestamps (fonte de verdade)
        F.when(
            (F.col("ch.status_consistencia") == "CONSISTENTE")
            & F.col("ch.hora_inicio_atendimento").isNotNull()
            & F.col("ch.hora_abertura_chamado").isNotNull(),
            (F.unix_timestamp(F.col("ch.hora_inicio_atendimento"))
             - F.unix_timestamp(F.col("ch.hora_abertura_chamado")))
        ).cast("int").alias("tempo_espera"),
        F.when(
            (F.col("ch.status_consistencia") == "CONSISTENTE")
            & F.col("ch.hora_finalizacao_atendimento").isNotNull()
            & F.col("ch.hora_inicio_atendimento").isNotNull(),
            (F.unix_timestamp(F.col("ch.hora_finalizacao_atendimento"))
             - F.unix_timestamp(F.col("ch.hora_inicio_atendimento")))
        ).cast("int").alias("tempo_atendimento"),
        # Tempos originais (para auditoria/comparacao)
        F.col("c.tempo_espera").alias("tempo_espera_original"),
        F.col("c.tempo_atendimento").alias("tempo_atendimento_original"),
        # Dimensoes derivadas de data
        F.to_date(F.col("ch.hora_abertura_chamado")).alias("data_abertura"),
        F.date_format(F.col("ch.hora_abertura_chamado"), "yyyy-MM").alias("mes_abertura"),
        F.year(F.col("ch.hora_abertura_chamado")).alias("ano_abertura"),
        F.month(F.col("ch.hora_abertura_chamado")).alias("mes_num"),
        F.dayofweek(F.col("ch.hora_abertura_chamado")).alias("dia_semana_num"),
        F.date_format(F.col("ch.hora_abertura_chamado"), "EEEE").alias("dia_semana"),
        F.hour(F.col("ch.hora_abertura_chamado")).alias("hora_abertura"),
        # Fatos
        F.col("cu.custo"),
        F.col("ps.nota_media").alias("nota_atendimento"),
        # Flags de qualidade
        F.col("cli.nome").isNull().alias("flag_sem_cliente"),
        F.col("cu.custo").isNull().alias("flag_sem_custo"),
        F.col("ps.nota_media").isNull().alias("flag_sem_pesquisa"),
        (F.col("c.id_atendente") == -1).alias("flag_sem_atendente"),
        (F.col("ch.status_consistencia") != "CONSISTENTE").alias("flag_ordem_invalida"),
    )
)

# Gravacao idempotente
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.ft_chamados")
)

print(f"Tabela {gold_schema}.ft_chamados criada/atualizada!")
print(f"Total de linhas: {spark.table(f'{gold_schema}.ft_chamados').count():,}")

## Bridge table — `bridge_chamado_pesquisa`

Preserva a relação **N:N** entre chamados e pesquisas de satisfação. 802 chamados têm múltiplas pesquisas; a tabela fato aggrega (`nota_media`), mas a bridge mantém as notas individuais para auditoria e análise de divergências.

In [0]:
# --- bridge_chamado_pesquisa ---
# N:N entre ft_chamados e pesquisas individuais (81.021 pesquisas, 802 chamados com múltiplas).
df_bridge = spark.table(f"{silver_schema}.tb_pesquisa_satisfacao").select(
    F.col("id_pesquisa"),
    F.col("id_chamado"),
    F.col("nota_atendimento"),
    F.col("nota_original"),
    F.col("flag_sem_nota"),
    F.col("flag_nota_fora_dominio"),
    F.col("flag_chamado_multiplas_pesquisas"),
)

df_bridge.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_chamado_pesquisa")
print(f"bridge_chamado_pesquisa: {df_bridge.count():,} linhas")

In [0]:
# === Validacao ===
print("=== Dimensões ===")
for dim in ["dim_cliente", "dim_atendente", "dim_canal", "dim_motivo", "dim_tempo"]:
    n = spark.table(f"{gold_schema}.{dim}").count()
    print(f"  {dim}: {n:,} linhas")

print("\n=== Fato ===")
df_gold = spark.table(f"{gold_schema}.ft_chamados")
print(f"  ft_chamados: {df_gold.count():,} linhas (esperado: 131.838)")

print("\n  Flags de qualidade:")
df_gold.select(
    F.sum(F.col("flag_sem_cliente").cast("int")).alias("sem_cliente"),
    F.sum(F.col("flag_sem_custo").cast("int")).alias("sem_custo"),
    F.sum(F.col("flag_sem_pesquisa").cast("int")).alias("sem_pesquisa"),
    F.sum(F.col("flag_sem_atendente").cast("int")).alias("sem_atendente"),
    F.sum(F.col("flag_ordem_invalida").cast("int")).alias("ordem_invalida"),
).show()

print("  Tempos (fonte de verdade = timestamps):")
df_gold.select(
    F.sum(F.col("tempo_espera").isNull().cast("int")).alias("espera_null"),
    F.sum(F.col("tempo_espera_original").isNull().cast("int")).alias("espera_orig_null"),
    F.sum(F.col("tempo_atendimento").isNull().cast("int")).alias("atendimento_null"),
    F.sum(F.col("tempo_atendimento_original").isNull().cast("int")).alias("atend_orig_null"),
).show()

print("\n=== Bridge ===")
df_bridge = spark.table(f"{gold_schema}.bridge_chamado_pesquisa")
print(f"  bridge_chamado_pesquisa: {df_bridge.count():,} linhas")
df_bridge.groupBy("flag_chamado_multiplas_pesquisas").count().show()

print("=== Integridade FKs ===")
# FK dim_motivo: todos os id_motivo devem existir na dim
fk_motivo = spark.sql(f"""
    SELECT COUNT(*) AS fk_motivo_orfao
    FROM {gold_schema}.ft_chamados f LEFT ANTI JOIN {gold_schema}.dim_motivo d ON f.id_motivo = d.id_motivo
    WHERE f.id_motivo IS NOT NULL
""")
fk_motivo.show()
# FK dim_canal
fk_canal = spark.sql(f"""
    SELECT COUNT(*) AS fk_canal_orfao
    FROM {gold_schema}.ft_chamados f LEFT ANTI JOIN {gold_schema}.dim_canal d ON f.id_canal = d.id_canal
    WHERE f.id_canal IS NOT NULL
""")
fk_canal.show()
# FK dim_cliente
fk_cli = spark.sql(f"""
    SELECT COUNT(*) AS fk_cliente_orfao
    FROM {gold_schema}.ft_chamados f LEFT ANTI JOIN {gold_schema}.dim_cliente d ON f.id_cliente = d.id_cliente
""")
fk_cli.show()
# FK dim_tempo
spark.sql(f"""
    SELECT COUNT(*) AS fk_tempo_orfao
    FROM {gold_schema}.ft_chamados f LEFT ANTI JOIN {gold_schema}.dim_tempo d ON f.data_abertura = d.data
""").show()